# Appendix E: sensitivity to the selected Top-k depth

Redraw the saved B results for k = 2, 3, 4 and 5 at background size K = 100. Each heatmap cell is **100 × count(distance > saved threshold) / (10N)**: ten unordered pairs from five explanations for every query, with a strict `>` comparison. It is not a percentage of queries or a threshold applied after averaging pair distances.

**Input:** `figures_b/background_topk_table/pair_exceedance_counts.csv` (128 rows). The main-table comparison reads `figures_b/background_table/pair_exceedance_counts.csv`. This notebook validates and visualizes saved summaries; it does not load raw SHAP arrays, rerun baselines, train models, or generate explanations. The historical query/model provenance is not revalidated by reading these summaries.

Run all cells with Python, NumPy, pandas, Matplotlib, IPython and Jupyter installed. The helper is `analysis/appendix_e/depth_sweep.py`; no machine-specific path is required.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

# Locate this checkout when run from its root or the notebook directory.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / ".shap_multiplicity_root").is_file()
             and (p / "figures_b/background_table/pair_exceedance_counts.csv").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Run from this repository or one of its subdirectories.")
OUTPUT = REPO / "appendix_e"
HELPER_ROOT = REPO / "analysis"
sys.path.insert(0, str(HELPER_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "appendix-e-matplotlib"))

import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from appendix_e.depth_sweep import (
    read_inputs, validate_topk, validate_sweep, export_analysis,
    plot_topk, plot_background_size, save_figure,
)
%matplotlib inline
print("Repository:", REPO)
print("Outputs:", OUTPUT)

## Validate the complete design and the saved B arithmetic

Require every dataset × model × method × depth combination, complete source status, finite values, integer numerator/denominator, and `n_observations = 10N`. Read decimal floats with round-trip precision. Compare k = 3 against all 32 matching KernelSHAP main-table rows, including counts, percentages and thresholds. Thresholds must remain fixed across methods/models within each dataset/depth.

In [ ]:
topk, main = read_inputs(REPO, "topk")
topk_report, topk_equivalence = validate_topk(topk, main)
print(json.dumps(topk_report, indent=2))
display(topk_equivalence[["dataset", "model", "method", "percent", "n_exceed", "n_observations", "threshold"]])

## Export the plotting table and its provenance

The portable plotting CSV retains all source columns and adds labels, units and the source path. The JSON sidecar records SHA-256 hashes of both source CSVs, the plotting CSV and validation results. No saved source is rewritten. Inspect the distinct thresholds below; each k has its own calibrated baseline.

In [ ]:
topk_provenance = export_analysis(REPO, OUTPUT, "topk", topk, topk_report, topk_equivalence)
topk_plot = pd.read_csv(OUTPUT / "data/topk_plot_data.csv", float_precision="round_trip")
display(topk_plot[["dataset_label", "k", "threshold", "minimum_positive_jaccard_distance"]]
        .drop_duplicates().reset_index(drop=True))
print("Exported:", topk_provenance["plot_data"]["path"])

## Draw all four depths on one common scale

Each panel contains eight dataset/model rows and the same four method columns. All panels share a 0–100% YlOrRd scale; annotations show one decimal place while the CSV retains the unrounded percentages. Dataset separators aid comparison. The k = 3 panel matches the main-table data exactly.

In [ ]:
fig = plot_topk(topk_plot)
topk_assets = save_figure(fig, "appendix_topk_depth")
plt.show()
plt.close(fig)
print(json.dumps(topk_assets, indent=2))

## Interpretation and limits

Here every stored threshold is below the smallest positive Top-k Jaccard distance, `2/(k+1)`. Consequently, these particular B percentages also equal Top-k set-disagreement frequencies. This equality is specific to these saved thresholds; it is not a general property of calibration or a significance test. Different k values change both the observed sets and the baseline, so an increase is not an increase in one fixed statistic. Top-1 any-seed churn is a separate statistic and is not included in this heatmap.